# 🚀 HUẤN LUYỆN FINE-TUNE `PP-OCRv6_small_rec` CHUYÊN BIỆT CHO BẢN VẼ KỸ THUẬT CAD

Notebook này được thiết kế chuẩn cho **Google Colab (GPU T4)** để fine-tune mô hình **`PP-OCRv6_small_rec`** (PPLCNetV4 + LightSVTR + MultiHead CTC/NRTR) trên tập dữ liệu **4,805 ảnh CAD đơn dòng**:

### 🎯 7 Nhóm kỹ năng CAD được huấn luyện (Không chèn dấu cách giả vào số):
1. **Độ DMS & Radian / Ngoặc đơn**: `0° 10' 36"` → `0°10'36"`, `15° 38' 32"` → `15°38'32"`, `(0.581°`, `(0.581°)`, `0.524rad`
2. **Vát mép `C` & Bán kính `R`**: `C0.02`, `C0.5`, `R0.9`, `C0.03MAX.`, `2-C0.02`
3. **Ký hiệu Phi & Đường kính (`Ø`, `ϕ`)**: `Ø12.5`, `ϕ8.0`, `Ø0.502`, `2-Ø5.5`, `Ø6.0H7` (cả nét mảnh SHX & nét đậm TrueType)
4. **Góc nghiêng (`±15°..±55°`)**: kèm đường gióng nghiêng, cung tròn kích thước và nét cắt
5. **Gạch chân sát chữ (Underline)** & chữ bị cắt mép dưới `(FLT)` / `(MAX)`
6. **Nhiễu scan / Mờ nhẹ / Nét đứt / Nét đậm đâm xuyên chữ**
7. **Giữ vững đọc số chuẩn (Clean Replay)**: `0.180`, `0.090`, `+0.005`, `-0.010`, `12.5±0.02` với `learning_rate = 0.00002` (25 epochs), giữ nguyên 100% từ điển gốc `ppocrv6_dict.txt` (18,708 ký tự) và trọng số Pretrained Head.

---
- **Đầu vào**: Gói `ppocrv6_small_rec_cad_pack.zip` (~33.9 MB)
- **Đầu ra**: File **`PP-OCRv6_small_rec_041026.onnx`** tự động tải về máy, sẵn sàng nạp vào Mini PDF OCR Tester.

## Bước 1: Kiểm tra GPU
Đảm bảo bạn đã chọn **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# BƯỚC 1: KIỂM TRA GPU TRÊN COLAB
!nvidia-smi


## Bước 2: Cài đặt siêu tốc PaddlePaddle GPU & PaddleOCR
Sử dụng **`aria2c` (16 luồng)** để tải wheel `paddlepaddle-gpu` với tốc độ ~70–100 MB/s và **`uv`** để cài đặt thư viện trong vài giây.

In [ ]:
# BƯỚC 2: CÀI ĐẶT PADDLEPADDLE GPU & PADDLEOCR (SIÊU TỐC BẰNG ARIA2C 16 LUỒNG + UV)
%cd /content
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q uv

# 1. Tải trực tiếp wheel paddlepaddle-gpu 3.3.0 (1.9GB) từ CDN bằng aria2c 16 luồng (~70-100 MB/s)
import os, sys
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl_name = f'paddlepaddle_gpu-3.3.0-{py_tag}-{py_tag}-linux_x86_64.whl'
whl_url = f'https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/{whl_name}'
!aria2c -s 16 -x 16 -k 1M -c "{whl_url}" -o "{whl_name}"
!uv pip install --system "{whl_name}"

# 2. Clone PaddleOCR (chứa đầy đủ kiến trúc PP-OCRv6_small_rec)
if not os.path.exists('/content/PaddleOCR_v6_cad/.git'):
    !git clone --depth 1 --single-branch https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR_v6_cad

# 3. Cài đặt các thư viện phụ trợ bằng uv (chỉ mất ~5 giây)
!uv pip install --system -r /content/PaddleOCR_v6_cad/requirements.txt pyclipper lmdb rapidfuzz shapely scikit-image

# 4. Xác nhận PaddlePaddle nhận diện GPU CUDA thành công
import paddle
print('====================================================')
print('🎉 PaddlePaddle version:', paddle.__version__)
print('🔥 GPU CUDA sẵn sàng   :', paddle.device.is_compiled_with_cuda())
assert paddle.device.is_compiled_with_cuda(), 'Lỗi: PaddlePaddle chưa nhận GPU CUDA!'
print('====================================================')


## Bước 3: Tải lên `ppocrv6_small_rec_cad_pack.zip` & Tải Pretrained Weights gốc
Nếu bạn đã upload file `ppocrv6_small_rec_cad_pack.zip` vào cột Files bên trái của Colab (`/content/`), cell này sẽ tự động nhận diện ngay mà không cần bấm chọn lại.

In [ ]:
# BƯỚC 3: UPLOAD & GIẢI NÉN DATASET + TẢI PRETRAINED WEIGHTS CHÍNH THỨC
%cd /content
import os, json, shutil, zipfile, hashlib
from pathlib import Path
from google.colab import files

ZIP_PATH = Path('/content/ppocrv6_small_rec_cad_pack.zip')
if not ZIP_PATH.is_file():
    print('👉 Vui lòng chọn file ppocrv6_small_rec_cad_pack.zip từ máy tính của bạn:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            shutil.move(fn, str(ZIP_PATH))
            break

assert ZIP_PATH.is_file(), 'Không tìm thấy file ppocrv6_small_rec_cad_pack.zip!'
print('📦 Đang giải nén ppocrv6_small_rec_cad_pack.zip...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/content')

BUNDLE = Path('/content/ppocrv6_small_rec_cad_pack')
WEIGHTS_DIR = BUNDLE / 'weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_PATH = WEIGHTS_DIR / 'PP-OCRv6_small_rec_pretrained.pdparams'

if not WEIGHTS_PATH.is_file():
    print('⬇️ Đang tải PP-OCRv6_small_rec_pretrained.pdparams từ Paddle CDN...')
    !aria2c -s 16 -x 16 -k 1M -c "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_rec_pretrained.pdparams" -d "/content/ppocrv6_small_rec_cad_pack/weights" -o "PP-OCRv6_small_rec_pretrained.pdparams"

meta = json.loads((BUNDLE / 'bundle_info.json').read_text(encoding='utf-8'))
actual_sha = hashlib.sha256(WEIGHTS_PATH.read_bytes()).hexdigest()
assert actual_sha == meta['checkpointSha256'], f'SHA256 không khớp: {actual_sha}'

print('✔ Giải nén Dataset & Xác thực SHA256 Pretrained thành công!')
print('📊 Thống kê bộ dữ liệu:', json.dumps(meta['datasetCounts'], indent=2, ensure_ascii=False))


## Bước 4: Tạo file cấu hình `rec_ppocrv6_small_cad.yml` & Kiểm tra khớp 100% Tensor Pretrained
Thiết lập cấu hình trực tiếp trong Notebook (giữ nguyên 18,708 ký tự từ điển gốc, `learning_rate = 0.00002`, `25 epochs`) và kiểm tra 100% các lớp mạng khớp tuyệt đối với trọng số Pretrained.

In [ ]:
# BƯỚC 4: TẠO CONFIG TRỰC TIẾP, KIỂM TRA TENSOR PRETRAINED & XEM BẢNG ẢNH MẪU
import copy, sys, yaml
from pathlib import Path
from IPython.display import display
from PIL import Image

EPOCHS = 25
BATCH = 32          # Giảm xuống 16 nếu muốn tiết kiệm VRAM
LR = 0.00002        # Learning rate êm để giữ vững 100% khả năng đọc số chuẩn

BUNDLE = Path('/content/ppocrv6_small_rec_cad_pack')
REPO = Path('/content/PaddleOCR_v6_cad')
DATASET = BUNDLE / 'dataset'

cfg = yaml.safe_load((REPO / 'configs/rec/PP-OCRv6/PP-OCRv6_small_rec.yml').read_text(encoding='utf-8'))
dict_src = REPO / 'ppocr/utils/dict/ppocrv6_dict.txt'
(BUNDLE / 'pretrained_dict.txt').write_text(dict_src.read_text(encoding='utf-8'), encoding='utf-8')

cfg['Global'].update(
    use_gpu=True,
    distributed=False,
    epoch_num=EPOCHS,
    pretrained_model=str(BUNDLE / 'weights/PP-OCRv6_small_rec_pretrained'),
    checkpoints=None,
    character_dict_path=str(BUNDLE / 'pretrained_dict.txt'),
    save_model_dir=str(BUNDLE / 'output/cad_singleline'),
    save_inference_dir=str(BUNDLE / 'inference/cad_singleline'),
    eval_batch_step=[0, 150],
    save_epoch_step=5,
    seed=20261004,
    use_amp=False,
)
cfg['Metric'].update(is_filter=False, ignore_space=False)
cfg['Optimizer']['lr']['learning_rate'] = LR
cfg['Optimizer']['lr']['warmup_epoch'] = 2

transforms = [
    {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
    {'MultiLabelEncode': {'gtc_encode': 'NRTRLabelEncode'}},
    {'RecResizeImg': {'image_shape': [3, 48, 320]}},
    {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
]
for key, split in [('Train', 'train'), ('Eval', 'val')]:
    cfg[key] = {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': str(DATASET),
            'label_file_list': [str(DATASET / f'rec_{split}.txt')],
            'transforms': transforms,
        },
        'loader': {
            'shuffle': key == 'Train',
            'batch_size_per_card': BATCH,
            'drop_last': key == 'Train',
            'num_workers': 2,
        },
    }

CONFIG_PATH = BUNDLE / 'rec_ppocrv6_small_cad.yml'
CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('✔ Đã lưu file cấu hình:', CONFIG_PATH)

# Kiểm tra khớp 100% Tensor giữa kiến trúc PP-OCRv6_small_rec và Pretrained checkpoint
sys.path.insert(0, str(REPO))
import paddle
from ppocr.modeling.architectures import build_model
from ppocr.postprocess import build_post_process

decoder = build_post_process(cfg['PostProcess'], cfg['Global'])
n_chars = len(decoder.character)
arch = copy.deepcopy(cfg['Architecture'])
arch['Head']['out_channels_list'] = {'CTCLabelDecode': n_chars, 'NRTRLabelDecode': n_chars + 3, 'SARLabelDecode': n_chars + 2}
model = build_model(arch)
weights = paddle.load(str(BUNDLE / 'weights/PP-OCRv6_small_rec_pretrained.pdparams'))
expected = model.state_dict()
mismatched = [k for k, v in expected.items() if k not in weights or list(v.shape) != list(weights[k].shape)]
assert not mismatched, f'Lỗi lệch Tensor Pretrained: {mismatched}'
print(f'✔ Xác thực thành công 100% ({len(expected)}/{len(expected)} tensors khớp hoàn toàn, {n_chars} ký tự)!')

display(Image.open(BUNDLE / 'preview_8_categories.png'))


## Bước 5: Tiến hành Fine-Tune `PP-OCRv6_small_rec` (25 Epochs ~ 8–10 phút)
Chạy trực tiếp bằng lệnh `!python -u` để hiển thị trực tiếp (`realtime`) từng dòng `epoch`, `iter`, `loss`, `acc` và `best_accuracy`.

In [ ]:
# BƯỚC 5: CHẠY FINE-TUNE PP-OCRv6_small_rec (HIỂN THỊ LOG TRỰC TIẾP)
%cd /content/PaddleOCR_v6_cad
!PYTHONUNBUFFERED=1 python -u tools/train.py -c /content/ppocrv6_small_rec_cad_pack/rec_ppocrv6_small_cad.yml

!ls -lh /content/ppocrv6_small_rec_cad_pack/output/cad_singleline/


## Bước 6: Đánh giá độ chính xác trên tập Validation
Kiểm tra điểm Accuracy của checkpoint tốt nhất (`best_accuracy`).

In [ ]:
# BƯỚC 6: ĐÁNH GIÁ ĐỘ CHÍNH XÁC CHECKPOINT TỐT NHẤT (best_accuracy)
%cd /content/PaddleOCR_v6_cad
!PYTHONUNBUFFERED=1 python -u tools/eval.py \
    -c /content/ppocrv6_small_rec_cad_pack/rec_ppocrv6_small_cad.yml \
    -o Global.checkpoints=/content/ppocrv6_small_rec_cad_pack/output/cad_singleline/best_accuracy


## Bước 7: Xuất sang ONNX (`PP-OCRv6_small_rec_041026.onnx`) & Tự động tải về máy
Xuất từ `best_accuracy` sang Paddle Inference, chuyển đổi sang `.onnx` (opset 14) và tự động tải file `.onnx` về máy.

In [ ]:
# BƯỚC 7: XUẤT MÔ HÌNH SANG ONNX & TỰ ĐỘNG TẢI VỀ MÁY
%cd /content/PaddleOCR_v6_cad
from pathlib import Path
from google.colab import files

BEST_CKPT = '/content/ppocrv6_small_rec_cad_pack/output/cad_singleline/best_accuracy'
INFER_DIR = '/content/ppocrv6_small_rec_cad_pack/inference/cad_singleline'
ONNX_OUT  = '/content/PP-OCRv6_small_rec_041026.onnx'

# 1. Xuất từ checkpoint tốt nhất sang Paddle Inference
!python -u tools/export_model.py \
    -c /content/ppocrv6_small_rec_cad_pack/rec_ppocrv6_small_cad.yml \
    -o Global.checkpoints={BEST_CKPT} \
       Global.pretrained_model={BEST_CKPT} \
       Global.save_inference_dir={INFER_DIR}

# 2. Chuyển đổi sang ONNX trong môi trường Python 3.12 sạch bằng uv (mất ~5 giây, tránh xung đột thư viện)
!uv venv --clear --python 3.12 /content/onnx_env
!uv pip install --python /content/onnx_env/bin/python paddlepaddle paddle2onnx onnx onnxruntime packaging

infer_path = Path(INFER_DIR)
model_fname = 'inference.json' if (infer_path / 'inference.json').is_file() else 'inference.pdmodel'
print(f'🔄 Đang chuyển đổi {model_fname} sang {ONNX_OUT}...')

!/content/onnx_env/bin/paddle2onnx \
    --model_dir {INFER_DIR} \
    --model_filename {model_fname} \
    --params_filename inference.pdiparams \
    --save_file {ONNX_OUT} \
    --opset_version 14 \
    --enable_onnx_checker True

# 3. Hiển thị dung lượng & Tải trực tiếp file ONNX về máy
onnx_file = Path(ONNX_OUT)
assert onnx_file.is_file(), 'Lỗi: Chưa tạo được file ONNX!'
print('====================================================')
print(f'✔ Xuất thành công: {onnx_file} ({round(onnx_file.stat().st_size / 1024**2, 2)} MB)')
print('👉 Sau khi tải về, copy file này vào thư mục models/onnx_ppocr/ của Mini Tool.')
print('====================================================')
files.download(str(onnx_file))
